# Project: Build your own experiment tracker 🧾
MLflow and Weights & Biases look like magic, but the core is tiny: **every run appends one record** (params, metrics, a hash of the data, the code version) to a log. Build that in 20 lines with JSON Lines, run a small hyper-parameter sweep, find the best run, and prove your results are reproducible.

Topic: [[Experiment Tracking and Reproducibility]], [[MLOps Overview]] · guide note: [[Project - Build Your Own Experiment Tracker]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import hashlib, json, pathlib, subprocess, numpy as np, pandas as pd
from datetime import datetime, timezone
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
LOG = pathlib.Path("data/runs.jsonl"); LOG.parent.mkdir(exist_ok=True)
def load_runs(path=LOG):
    return [json.loads(l) for l in open(path, encoding="utf-8")] if pathlib.Path(path).exists() else []
def code_version():
    try: return subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=5).stdout.strip() or "unknown"
    except Exception: return "unknown"
X, y = load_digits(return_X_y=True)
data_hash = hashlib.sha256(X.tobytes() + y.tobytes()).hexdigest()[:12]          # fingerprint: same data ⇒ same hash
print("data fingerprint:", data_hash, "| code version:", code_version())

## 1. Log a run
Append **one JSON line** to `path` with: `run_id` (1 + number of existing runs), `time` (UTC ISO), `params`, `metrics`, `data_hash` and `code_version()`. Return the record.

In [ ]:
def log_run(params, metrics, path=LOG):
    # TODO 1: build the record dict and append it as one JSON line
    raise NotImplementedError  # TODO 1

## 2. Find the best run
Among runs that have `metric`, return the one with the highest value (or lowest if `higher_is_better=False`).

In [ ]:
def best_run(runs, metric, higher_is_better=True):
    # TODO 2: filter runs with the metric, then max/min by it
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_track():
    tmp = pathlib.Path("data/_test_runs.jsonl"); tmp.unlink(missing_ok=True)
    a = log_run({"lr": 1}, {"acc": 0.8}, tmp); b_ = log_run({"lr": 2}, {"acc": 0.9, "loss": 0.3}, tmp)
    runs = load_runs(tmp); tmp.unlink()
    return [r["run_id"] for r in runs] == [1, 2] and runs[1]["params"] == {"lr": 2} and "data_hash" in runs[0] and \
        best_run(runs, "acc")["run_id"] == 2 and best_run(runs, "loss", False)["run_id"] == 2
check("tracker", _t_track, "Two runs → ids 1 and 2, params/metrics/data_hash stored; best by acc is run 2.")

def train_eval(hidden, alpha, seed):
    Xtr, Xte, ytr, yte = train_test_split(X / 16, y, test_size=0.3, random_state=0)
    m = MLPClassifier(hidden_layer_sizes=(hidden,), alpha=alpha, max_iter=300, random_state=seed).fit(Xtr, ytr)
    return m.score(Xte, yte)

if ready("tracker"):
    LOG.unlink(missing_ok=True)
    for hidden in (16, 64):
        for alpha in (1e-4, 1e-2, 1.0):
            acc = train_eval(hidden, alpha, seed=0); log_run({"hidden": hidden, "alpha": alpha, "seed": 0}, {"test_acc": acc})
    runs = load_runs(); table = pd.DataFrame([{**r["params"], **r["metrics"], "run": r["run_id"]} for r in runs]); print(table.to_string(index=False))
    best = best_run(runs, "test_acc"); print("\nbest:", best["params"], best["metrics"])
    again = train_eval(best["params"]["hidden"], best["params"]["alpha"], best["params"]["seed"])
    other_seeds = [train_eval(best["params"]["hidden"], best["params"]["alpha"], s) for s in (1, 2, 3)]
    print(f"re-run with the logged seed: {again:.4f} (logged {best['metrics']['test_acc']:.4f}); other seeds: {np.round(other_seeds, 4)}")
    check("reproducible", lambda: again == best["metrics"]["test_acc"], "Re-running the best config with the logged seed must give exactly the logged score.")
    print("→ The spread across seeds is your noise floor: a 'better' config that wins by less than that hasn't won.")

<details><summary>▶ Solution</summary>

```python
def log_run(params, metrics, path=LOG):
    record = {"run_id": len(load_runs(path)) + 1, "time": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "params": params, "metrics": metrics, "data_hash": data_hash, "code_version": code_version()}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record
```

```python
def best_run(runs, metric, higher_is_better=True):
    runs = [r for r in runs if metric in r["metrics"]]
    return max(runs, key=lambda r: r["metrics"][metric]) if higher_is_better else min(runs, key=lambda r: r["metrics"][metric])
```
</details>

## Stretch goals
- Log artefacts too: save each model with `joblib` under `data/models/<run_id>.joblib` and store the path in the record.
- Add a `tags` field and a tiny query helper ("all runs with hidden=64 sorted by accuracy").
- Swap in MLflow (`mlflow.log_params`, `mlflow.log_metrics`) and compare what it stores beyond yours.